# TM-RugPull dataset initial analysis

## Data collection

## Data collection

In [ ]:
# Loading data from .xlsx file

import pandas as pd
import numpy as np
import matplotlib.pyplot as pyplot

file = 'data/TM-RugPull.xlsx'

data = pd.read_excel(file)

# Remove a space in the end of some column names
data.columns = data.columns.str.strip()

print(data.shape)

pd.set_option('display.max_columns', None)

data.head(5)

In [ ]:
# Check the balance between classes in the whole dataset
# Source: https://note.nkmk.me/en/python-pandas-value-counts/#value_counts

class_counts = data['class'].value_counts()
class_percent = data['class'].value_counts(normalize=True) * 100

print("Counts:", class_counts.to_string(), "\nIn %:", class_percent.to_string())

##### Create a test set and a validation test

In [ ]:
# Check general info about data

print("\nDataset information:")
data.info()

In [ ]:
# Create a test set and a validation set from the raw data to avoid data leakage
# Validation set size is about 10,5% and test set size is about 24,5% from the whole data set
# TODO reference to AML tutorial

from sklearn.model_selection import train_test_split

# define size of data both for test and validatioin sets, define the seed for all subsequent experiments
test_and_val_size = 0.35
seed = 7

# Split the data first on train set and set for test and validation
train_set, test_and_val_set = train_test_split(data, test_size=test_and_val_size, random_state=seed, stratify=data['class'])

# Split the part for test and validation into test set and validation set
test_set, val_set = train_test_split(test_and_val_set, test_size=0.3, random_state=seed, stratify=test_and_val_set['class'])

# Output the shapes to verify the splits
print("Training set shape:", train_set.shape)
print("Test set shape:", test_set.shape)
print("Validation set shape:", val_set.shape)

# Create a list of sets to perform further feature engeneering on all subsets of data
data_sets = [train_set, test_set, val_set]

In [ ]:
# Verify that there is no overlap between sets, no data leak at this stage
print("Overlap between train and test:", np.intersect1d(train_set.index, test_set.index).size)
print("Overlap between train and validation:", np.intersect1d(train_set.index, val_set.index).size)
print("Overlap between test and validation:", np.intersect1d(test_set.index, val_set.index).size)

In [ ]:
# Delete columns that are not useful for further analysis

for set in data_sets:
    set.drop(columns=['Project Title', 'Sign', 'website', 'x profile', 'Smart Contract (online)', 'smart Contract (offline)', 'project starting date', 'project end date'], inplace=True)

# Check the transformation
train_set.head(10)

## Data analysis

### Performed on raw data to get overall idea of what the dataset contains

In [ ]:
# Encode class label

from sklearn.preprocessing import LabelEncoder

for set in data_sets:
    le = LabelEncoder()
    set['class'] = set['class'].map({'normal': 0, 'scam': 1})

train_set

In [ ]:
# Check the data description

description = train_set.describe()
description

In [ ]:
# Check the balance between classes in the training set

class_counts_train = data['class'].value_counts()
class_percent_train = data['class'].value_counts(normalize=True) * 100

print("Counts:", class_counts_train.to_string(), "\nIn %:", class_percent_train.to_string())

In [ ]:
print(train_set.dtypes)

In [ ]:
# Transfer all data to numeric values

#TODO: reference from notes

# Columns with object datatype
cols_to_clean = [
    'the number of Transactions',
    'Token concentration ratio per holder',
    'Token balance',
    'first deposits',
    'Google results for project website (first day)',
    'Google results for project website (duration/2)'
]

# Do it for all sets
data_sets = [train_set, test_set, val_set]

for i, d_set in enumerate(data_sets):
    for col in cols_to_clean:
        data_sets[i][col] = pd.to_numeric(
            d_set[col].astype(str)
                   .str.replace('\xa0', '', regex=False)        # Remove non-breaking whitespaces
                   .str.replace(',', '', regex=False)           # Strip comas separating numeric values
                   .str.strip(),                                # Remove surrounding whitespaces
            errors='coerce'                                     # If cannot parse, put NaN
        )

train_set, test_set, val_set = data_sets

# Verify
print(train_set[cols_to_clean].dtypes)
print(train_set[cols_to_clean].isnull().sum())
print(train_set[cols_to_clean].describe())

In [ ]:
# Check for skew for numeric columns

numeric_cols = train_set.select_dtypes(include=['number']).columns
skew = train_set[numeric_cols].skew()

skew

### Visualisation

##### Based on CSM010-2024-APR, Topic 2, Lab.2.15. Analysing data

In [ ]:
# Histograms

train_set.hist(figsize=[30, 30])
pyplot.show()

In [ ]:
# Density plots

train_set.plot(kind='density', subplots=True, layout=(8,7), sharex=False, sharey=False, figsize=[30, 30])
pyplot.show()

In [ ]:
# Box and Whisker Plots

train_set.plot(kind='box', subplots=True, layout=(8,7), sharex=False, sharey=False,figsize=[30, 30])
pyplot.show()

In [ ]:
# Search for correlations of numeric features

correlations = train_set[numeric_cols].corr(method='pearson')

correlations

In [ ]:
# Correlation Matrix Plot
# TODO: reference to AML tutorial

fig = pyplot.figure(figsize=[30, 30])
ax = fig.add_subplot(111)
cax = ax.matshow(correlations, vmin=-1, vmax=1)
fig.colorbar(cax)

ticks = np.arange(len(correlations.columns))

ax.set_xticks(ticks)
ax.set_yticks(ticks)

short_names = list(train_set[numeric_cols].columns)

ax.set_xticklabels(short_names)
ax.set_yticklabels(short_names)

pyplot.xticks(rotation=90)     # https://www.geeksforgeeks.org/how-to-rotate-x-axis-tick-label-text-in-matplotlib/?ysclid=lxdkiwmkrh456759424

pyplot.show()

In [ ]:
# Scatter plot Matrix
# TODO: make it more readable

pd.plotting.scatter_matrix(data, figsize=[20, 20])

pyplot.show()

## Pre-processing of data

### Based on results of data analysis on raw data

In [ ]:
#TODO: analyse non-numeric features?? Think of ways, read

In [ ]:
# Replace strings in 'Blockchain', 'Blockchain Type', 'class' columns with numbers
# to make all features numeric for further analysis

#TODO: should I enforce specific values to each type of blockchain and its type or rely on LabelEncoder embedded?
#TODO: other ways to encode class values (see the book)
#TODO: reference to AML tutorial

from sklearn.preprocessing import LabelEncoder

for set in data_sets:
    le = LabelEncoder()
    set['Blockchain'] = le.fit_transform(set['Blockchain'])
    set['Blockchain Type'] = le.fit_transform(set['Blockchain Type'])

train_set

In [ ]:
# TODO: data require preparation and pre-processing (heavy skew, issues with 'first deposit' column (it seems that it is incorrect, duplicating Q1), outliers + a couple of missing values

# TODO: preparation and pre-processing